# WTI Price Return Prediction — GRU Model
Predicts weekly WTI price return (`close_pct_change`) from lagged news features using a Gated Recurrent Unit network.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

# ── Walk-forward parameters ────────────────────────────────────────────────────
INITIAL_TRAIN_WEEKS  = 104   # ~2 years minimum before first OOS fold
STEP_WEEKS           = 13    # quarterly test windows
CONFIDENCE_THRESHOLD = 1.0

# ── Model hyperparameters ──────────────────────────────────────────────────────
LOOKBACK    = 4       # weeks of history fed to GRU per prediction
HIDDEN_SIZE = 32      # GRU hidden units
NUM_LAYERS  = 1
DROPOUT     = 0.3
BATCH_SIZE  = 16
MAX_EPOCHS  = 300
PATIENCE    = 25      # early stopping patience (val Huber loss)
LR          = 1e-3

# ── Normalisation / weighting ──────────────────────────────────────────────────
HUBER_DELTA = 0.5
DECAY_RATE  = 0.995   # per-week exponential sample weight decay
ROLL_WINDOW = 52      # rolling z-score window (weeks)
VOL_WINDOW  = 8       # rolling vol normalisation window (weeks)

FEATURE_COLS = [
    # ── News features ─────────────────────────────────────────────────────────
    'headline_count_lag1',
    'sentiment_mean_lag1',
    'sentiment_std_lag1',
    'sentiment_min_lag1',
    'sentiment_max_lag1',
    'negative_ratio_lag1',
    'positive_ratio_lag1',
    'log_headline_count_lag1',
    'opec_event_day_lag1',
    'opec_decision_day_lag1',
    'opec_sentiment_lag1',
    'disruption_event_day_lag1',
    'disruption_intensity_lag1',
    # ── FinBERT probability features ──────────────────────────────────────────
    'finbert_positive_mean_lag1',
    'finbert_negative_mean_lag1',
    'finbert_neutral_mean_lag1',
    # ── Regime features ───────────────────────────────────────────────────────
    'realized_vol_lag1',
    'regime_0_lag1',
    'regime_1_lag1',
    'regime_2_lag1',
    'hmm_regime_0_lag1',
    'hmm_regime_1_lag1',
    'hmm_regime_2_lag1',
    # ── EIA macro features ────────────────────────────────────────────────────
    'inventory_chg_kb_lag1',
    'inventory_surprise_kb_lag1',
    'production_chg_kbd_lag1',
    'imports_chg_kbd_lag1',
    'refinery_util_pct_lag1',
    'refinery_util_chg_pct_lag1',
    # ── FRED macro features ───────────────────────────────────────────────────
    'usd_ret_pct_lag1',
    'usd_mom_4w_lag1',
    'spx_ret_pct_lag1',
    'yield_10y_chg_lag1',
    # ── Paper features: decay sentiment + intensity-weighted variance ─────────
    'broad_cs_di_lag1',
    'opec_cs_di_lag1',
    'disruption_cs_di_lag1',
    'broad_csi_v_lag1',
    'opec_csi_v_lag1',
    'disruption_csi_v_lag1',
]

REGIME_ONEHOT_COLS = [c for c in FEATURE_COLS if
                      c.startswith('regime_') or c.startswith('hmm_regime_')
                      or c.startswith('finbert_')]

TARGET_COL = 'close_pct_change'


## 2. Load & Inspect Data

In [ ]:
df = pd.read_parquet('../data/features/feature_matrix.parquet')
df = df.sort_values('date').reset_index(drop=True)

print(f'Rows: {len(df)}  |  Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df[FEATURE_COLS + [TARGET_COL]].describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(df['date'], df['close'], color='steelblue')
axes[0].set_title('WTI Spot Price ($/bbl)')
axes[0].set_ylabel('Price')
axes[1].plot(df['date'], df[TARGET_COL], color='darkorange', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Weekly Return (%) — Target Variable')
axes[1].set_ylabel('% Change')
plt.tight_layout()
plt.show()

## 3. Sequence Construction & Train/Test Split

In [ ]:
# ── Rolling z-score normalisation (backward-looking, no look-ahead) ───────────
scaled_df = df[FEATURE_COLS].copy().astype(np.float32)
for col in FEATURE_COLS:
    if col in REGIME_ONEHOT_COLS:
        continue
    roll_mean = scaled_df[col].rolling(ROLL_WINDOW, min_periods=8).mean()
    roll_std  = scaled_df[col].rolling(ROLL_WINDOW, min_periods=8).std().clip(lower=1e-6)
    scaled_df[col] = (scaled_df[col] - roll_mean) / roll_std

scaled_df = scaled_df.fillna(0.0)
features  = np.nan_to_num(scaled_df.values.astype(np.float32), nan=0.0)

# ── Volatility-normalised target ───────────────────────────────────────────────
target_raw = np.nan_to_num(df[TARGET_COL].values.astype(np.float32), nan=0.0)

rolling_vol = (
    df[TARGET_COL]
    .rolling(VOL_WINDOW, min_periods=4)
    .std()
    .clip(lower=0.5)
    .bfill()
    .values.astype(np.float32)
)

target_norm = np.clip(target_raw / rolling_vol, -10, 10)

# ── Build (X, y) sequences ────────────────────────────────────────────────────
X, y, dates, vols, raw_returns = [], [], [], [], []
for i in range(LOOKBACK, len(df)):
    X.append(features[i - LOOKBACK:i])
    y.append(target_norm[i])
    dates.append(df['date'].iloc[i])
    vols.append(rolling_vol[i])
    raw_returns.append(target_raw[i])    # actual % return at prediction date

X           = np.array(X)
y           = np.array(y)
dates       = np.array(dates)
vols        = np.array(vols, dtype=np.float32)
raw_returns = np.array(raw_returns, dtype=np.float32)

assert not np.isnan(X).any() and not np.isnan(y).any()

n_folds = (len(X) - INITIAL_TRAIN_WEEKS) // STEP_WEEKS
print(f'Total sequences : {len(X)}  ({dates[0]} → {dates[-1]})')
print(f'First OOS date  : {dates[INITIAL_TRAIN_WEEKS]}  (after {INITIAL_TRAIN_WEEKS}-week min training)')
print(f'Expected folds  : ~{n_folds}  (step={STEP_WEEKS} weeks each)')
print(f'Input shape     : ({INITIAL_TRAIN_WEEKS}, {LOOKBACK}, {len(FEATURE_COLS)})')


## 4. Dataset & DataLoader

In [ ]:
class WTIDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        if weights is not None:
            self.w = torch.tensor(weights, dtype=torch.float32)
        else:
            self.w = torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


## 5. GRU Model

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

# Preview parameter count
_probe = GRUModel(len(FEATURE_COLS), HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
print(f'GRUModel — parameters: {sum(p.numel() for p in _probe.parameters()):,}')
del _probe


## 6. Walk-Forward Training
Each fold trains a fresh GRU on all history to date (expanding window) with early
stopping (patience={PATIENCE}) on an internal validation split. Predictions are
accumulated across all folds for pooled OOS evaluation.


In [ ]:
criterion = nn.HuberLoss(reduction='none', delta=HUBER_DELTA)

fold_records = []
all_preds    = []
last_model   = None

for fold, test_start in enumerate(range(INITIAL_TRAIN_WEEKS, len(X), STEP_WEEKS)):
    test_end = min(test_start + STEP_WEEKS, len(X))
    if test_end - test_start < 5:
        break

    # ── Expanding training window ─────────────────────────────────────────────
    X_tr_full = X[:test_start]
    y_tr_full = y[:test_start]

    # Internal validation: last 10% of training sequences (chronological)
    val_n   = max(10, int(len(X_tr_full) * 0.10))
    X_tr, X_val = X_tr_full[:-val_n], X_tr_full[-val_n:]
    y_tr, y_val = y_tr_full[:-val_n], y_tr_full[-val_n:]

    X_te       = X[test_start:test_end]
    vols_te    = vols[test_start:test_end]
    act_raw    = raw_returns[test_start:test_end]
    dates_te   = dates[test_start:test_end]

    # ── Exponential sample weights ────────────────────────────────────────────
    n     = len(X_tr)
    raw_w = np.array([DECAY_RATE ** (n - 1 - i) for i in range(n)], dtype=np.float32)
    sw    = raw_w / raw_w.mean()

    train_ds  = WTIDataset(X_tr, y_tr, sw)
    val_ds    = WTIDataset(X_val, y_val)
    train_ldr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    # ── Fresh model each fold ─────────────────────────────────────────────────
    model_fold = GRUModel(len(FEATURE_COLS), HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
    optimizer  = torch.optim.Adam(model_fold.parameters(), lr=LR, weight_decay=1e-3)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)

    best_val      = float('inf')
    best_state    = None
    patience_cnt  = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model_fold.train()
        for xb, yb, wb in train_ldr:
            optimizer.zero_grad()
            raw_loss = criterion(model_fold(xb), yb)
            loss     = (raw_loss * wb.unsqueeze(1)).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model_fold.parameters(), 1.0)
            optimizer.step()

        model_fold.eval()
        with torch.no_grad():
            val_pred = model_fold(torch.tensor(X_val, dtype=torch.float32))
            val_loss = nn.HuberLoss(delta=HUBER_DELTA)(
                val_pred,
                torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
            ).item()
        scheduler.step(val_loss)

        if val_loss < best_val - 1e-5:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model_fold.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                break

    model_fold.load_state_dict(best_state)

    # ── OOS predictions ───────────────────────────────────────────────────────
    model_fold.eval()
    with torch.no_grad():
        preds_norm = model_fold(torch.tensor(X_te, dtype=torch.float32)).squeeze().numpy()

    preds_pct = preds_norm * vols_te
    dir_acc_f = float(np.mean(np.sign(preds_pct) == np.sign(act_raw)))
    mae_f     = float(mean_absolute_error(act_raw, preds_pct))

    fold_records.append({
        'fold':       fold,
        'train_seqs': len(X_tr_full),
        'test_start': pd.Timestamp(dates_te[0]).date(),
        'test_end':   pd.Timestamp(dates_te[-1]).date(),
        'n_test':     len(act_raw),
        'mae':        mae_f,
        'dir_acc':    dir_acc_f,
        'stop_epoch': epoch - PATIENCE,
    })

    for i in range(len(act_raw)):
        all_preds.append({
            'fold':   fold,
            'date':   pd.Timestamp(dates_te[i]),
            'actual': float(act_raw[i]),
            'pred':   float(preds_pct[i]),
        })

    last_model = model_fold
    print(f'Fold {fold:2d}  train={len(X_tr_full):3d}  ',
          f'test={pd.Timestamp(dates_te[0]).date()} → {pd.Timestamp(dates_te[-1]).date()}  ',
          f'dir_acc={dir_acc_f:.1%}  best_val={best_val:.4f}  epochs={epoch-PATIENCE}')

folds_df = pd.DataFrame(fold_records)
results  = pd.DataFrame(all_preds).sort_values('date').reset_index(drop=True)
print(f'\nTotal OOS weeks: {len(results)}  across {len(folds_df)} folds')


## 7. Evaluation

In [ ]:
actual_all = results['actual'].values
pred_all   = results['pred'].values

mae_all  = mean_absolute_error(actual_all, pred_all)
rmse_all = np.sqrt(mean_squared_error(actual_all, pred_all))
dir_all  = np.mean(np.sign(pred_all) == np.sign(actual_all))

print('Walk-Forward OOS Summary (GRU)')
print('=' * 52)
print(f'Total OOS weeks      : {len(results)}  across {len(folds_df)} folds')
print(f'Date range           : {results["date"].min().date()} → {results["date"].max().date()}')
print()
print(f'MAE (all folds)      : {mae_all:.4f}%')
print(f'RMSE (all folds)     : {rmse_all:.4f}%')
print(f'Dir Acc (all folds)  : {dir_all:.1%}')
print(f'Variance ratio       : {pred_all.std() / actual_all.std():.3f}')
print('=' * 52)

print(f'\n{"Threshold":>12}  {"N kept":>6}  {"Dir Acc":>8}  {"Long":>6}  {"Short":>6}')
print('-' * 52)
for thresh in [0.0, 0.5, 1.0, 1.5, 2.0]:
    mask   = np.abs(pred_all) >= thresh
    n_kept = mask.sum()
    if n_kept < 5:
        break
    da      = np.mean(np.sign(pred_all[mask]) == np.sign(actual_all[mask]))
    n_long  = (pred_all[mask] > 0).sum()
    n_short = (pred_all[mask] < 0).sum()
    print(f'{thresh:>11.1f}%  {n_kept:>6}  {da:>8.1%}  {n_long:>6}  {n_short:>6}')

print('\nPer-fold directional accuracy:')
print(folds_df[['fold','test_start','test_end','n_test','mae','dir_acc','stop_epoch']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['darkorange' if d >= 0.55 else 'steelblue' if d >= 0.50 else 'salmon'
          for d in folds_df['dir_acc']]
ax.bar(folds_df['fold'], folds_df['dir_acc'], color=colors, edgecolor='white')
ax.axhline(0.50, color='black', linewidth=1, linestyle='--', label='50% baseline')
ax.axhline(dir_all, color='red', linewidth=1.5, label=f'Overall {dir_all:.1%}')
ax.set_xlabel('Fold'); ax.set_ylabel('Directional Accuracy')
ax.set_title('GRU Walk-Forward Directional Accuracy per Fold')
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for _, row in folds_df.iterrows():
    for ax in axes:
        ax.axvspan(pd.Timestamp(row['test_start']), pd.Timestamp(row['test_end']),
                   alpha=0.04, color='grey')

axes[0].plot(results['date'], results['actual'], label='Actual',    color='steelblue')
axes[0].plot(results['date'], results['pred'],   label='Predicted', color='darkorange', linestyle='--', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5, linestyle=':')
axes[0].axhline( CONFIDENCE_THRESHOLD, color='green', linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].axhline(-CONFIDENCE_THRESHOLD, color='red',   linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].set_title('GRU Walk-Forward — Predicted vs Actual Weekly Return (%)')
axes[0].set_ylabel('% Change'); axes[0].legend()

# Directional accuracy rolling 26-week window
roll_dir = pd.Series(
    (np.sign(results['pred'].values) == np.sign(results['actual'].values)).astype(float)
).rolling(26, min_periods=13).mean()
axes[1].plot(results['date'], roll_dir, color='steelblue', label='26-week rolling dir acc')
axes[1].axhline(0.5, color='black', linewidth=1, linestyle='--', label='50% baseline')
axes[1].axhline(dir_all, color='red', linewidth=1.5, linestyle='-', label=f'Overall {dir_all:.1%}')
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Directional Accuracy')
axes[1].set_title('Rolling 26-Week Directional Accuracy')
axes[1].legend()

plt.tight_layout(); plt.show()


In [ ]:
test_df = results.copy().sort_values('date').reset_index(drop=True)

test_df['signal_naive']    = np.where(test_df['pred'] > 0, 1, 0)
test_df['signal_filtered'] = np.where(
    test_df['pred'] >=  CONFIDENCE_THRESHOLD,  1,
    np.where(
    test_df['pred'] <= -CONFIDENCE_THRESHOLD, -1, 0))

test_df['return_naive']    = test_df['signal_naive']    * test_df['actual']
test_df['return_filtered'] = test_df['signal_filtered'] * test_df['actual']

test_df['cum_buyhold']  = (1 + test_df['actual']          / 100).cumprod()
test_df['cum_naive']    = (1 + test_df['return_naive']    / 100).cumprod()
test_df['cum_filtered'] = (1 + test_df['return_filtered'] / 100).cumprod()

filt_mask = test_df['signal_filtered'] != 0
n_trades  = filt_mask.sum()
n_long    = (test_df['signal_filtered'] ==  1).sum()
n_short   = (test_df['signal_filtered'] == -1).sum()
dir_filt  = float(np.mean(
    np.sign(test_df.loc[filt_mask, 'pred']) == np.sign(test_df.loc[filt_mask, 'actual'])
)) if filt_mask.any() else float('nan')

print(f'Walk-Forward Strategy Summary  ({test_df["date"].min().date()} → {test_df["date"].max().date()})')
print(f'OOS weeks                 : {len(test_df)}')
print(f'Confidence threshold      : |pred| >= {CONFIDENCE_THRESHOLD}%')
print(f'Trades taken              : {n_trades} / {len(test_df)}  ({n_long} long, {n_short} short)')
print(f'Directional acc (filtered): {dir_filt:.1%}')
print()
print(f'Final cumulative return:')
print(f'  Buy & Hold : {test_df["cum_buyhold"].iloc[-1]:.3f}x')
print(f'  Naive      : {test_df["cum_naive"].iloc[-1]:.3f}x')
print(f'  Filtered   : {test_df["cum_filtered"].iloc[-1]:.3f}x')

fig, ax = plt.subplots(figsize=(14, 5))
for _, row in folds_df.iterrows():
    ax.axvspan(pd.Timestamp(row['test_start']), pd.Timestamp(row['test_end']),
               alpha=0.04, color='grey')
ax.plot(test_df['date'], test_df['cum_buyhold'],  label='Buy & Hold',               color='steelblue', linestyle='--')
ax.plot(test_df['date'], test_df['cum_naive'],    label='Naive (long if pred > 0)', color='grey',      linestyle=':')
ax.plot(test_df['date'], test_df['cum_filtered'], label=f'Filtered (|pred|≥{CONFIDENCE_THRESHOLD}%)', color='darkorange')
ax.axhline(1, color='black', linewidth=0.5, linestyle=':')
ax.set_title('Cumulative Return: GRU Walk-Forward Strategy (full OOS period)')
ax.set_ylabel('Growth of $1'); ax.legend()
plt.tight_layout(); plt.show()
